![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 38 -- Lab 1: Two More Classes

A company runs a classifier that sorts customer texts into 5 classes. The texts are written in a language nobody on the team can read, and the classifier never reads them either: it reads, for every text, the 768 numbers that multilingual BERT (mBERT) gives it, its pooled output. Today the client wants 7 classes and has labelled examples of the 2 new ones. Your goal is a 7-class answer for 700 new texts: first under today's rule (the deployed classifier cannot change and nothing new is learned), then with the classifier trained further. Your file is scored with macro F1 over the 7 classes.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The vectors of the texts, never the texts. The setup cell leaves four files in `DATA_DIR`: `train.pt` (a dictionary of tensors: `id`, `vectors`, one row of 768 numbers per text, and `label`, a class from 0 to 6), `test.pt` (`id` and `vectors` of the 700 texts you classify), `classifier.pt` (the weights of the deployed classifier, an `nn.Linear(768, 5)` for the classes 0 to 4) and `sample_submission.csv` (the format of the file you hand in). A `.pt` file is read with `torch.load(path)`.

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [2]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [3]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "S14-6WXC")
ck.start(lab="nlp-03", student_id=STUDENT_ID)


Session open. Minute 4. Lab: Two More Classes.
This lab records cell runs, cell source and errors for assessment.
Parts 0 of 8: LOAD - | LOOK - | SPLIT - | BATCH - | MODEL - | LOOP - | EVAL - | SUBMIT -

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

torch.manual_seed(0)
print("torch", torch.__version__)


torch 2.14.1+cpu


In [5]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/two-more-classes")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


Using Colab cache for faster access to the 'two-more-classes' dataset.
Data folder: /kaggle/input/two-more-classes
['.nfs0000000011a4ec4a0000000a', 'classifier.pt', 'sample_submission.csv', 'test.pt', 'train.pt']


---
# The starter solution

Every olympiad task comes with a starter solution: a short notebook that reads the files and hands in a valid file. Its score is the **baseline**, and your points count up from it. This is the starter of this task. Read it before you begin; you may reuse any of its lines.

```python
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

test = torch.load(os.path.join(DATA_DIR, "test.pt"))
head = nn.Linear(768, 5)
head.load_state_dict(torch.load(os.path.join(DATA_DIR, "classifier.pt")))
head.eval()
with torch.no_grad():
    old = head(test["vectors"]).argmax(dim=1).numpy()      # the deployed classifier: a class from 0 to 4

# Draw a class from 0 to 6 for every text: when 5 or 6 comes up, say it; otherwise keep the classifier's answer.
rng = np.random.default_rng(0)
draw = rng.integers(0, 7, size=len(old))
labels = np.where(draw >= 5, draw, old)
submission = pd.DataFrame({"id": test["id"].numpy(), "label": labels})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
```

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the numbers before you build anything.

## Task 1: Load the files

Read `train.pt` and `test.pt` from `DATA_DIR` with `torch.load` into `train` and `test`, and `sample_submission.csv` with `pd.read_csv` into `sample`. Then name the tensors you will use: `X_train` (the vectors of `train`), `y_train` (its labels) and `X_test` (the vectors of `test`). Print the shapes of `X_train`, `y_train`, `X_test` and `sample`.

Syntax hint (the shape of the call, not the answer):

```python
table = torch.load(os.path.join(DATA_DIR, "name.pt"))      # a dictionary of tensors
```

In [8]:
train = torch.load(os.path.join(DATA_DIR, "train.pt"))
test = torch.load(os.path.join(DATA_DIR, "test.pt"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
X_train,x_test

X_train shape: torch.Size([2473, 768])
y_train shape: torch.Size([2473])
X_test shape: torch.Size([700, 768])
sample shape: (700, 2)


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [9]:
ck.load(train=X_train, test=X_test, sample=sample)


✓ LOAD: Your data is loaded and has the expected size.
Parts 1 of 8: LOAD ✓ | LOOK - | SPLIT - | BATCH - | MODEL - | LOOP - | EVAL - | SUBMIT -

## Task 2: Look at the numbers

Store the first five vectors of `X_train` in `first_rows` and the number of training texts in `n_rows`. Print how many texts each class has, and the smallest and the largest number in `X_train`.

Syntax hint (the shape of the call, not the answer):

```python
counts = pd.Series(tensor.numpy()).value_counts().sort_index()
```

In [14]:
first_rows = X_train.head()
n_rows = len(X_train)


first_rows: tensor([[-0.1391,  0.8480, -0.7045,  ..., -0.4990, -0.8969, -0.7742],
        [-0.5181,  0.0884, -0.4772,  ...,  0.7813,  0.5746, -0.3365],
        [-0.1337,  0.8543, -0.6965,  ..., -0.4501, -0.8997, -0.7674],
        [-0.1649,  0.8538, -0.6712,  ..., -0.3268, -0.8995, -0.7897],
        [-0.1198,  0.8906, -0.6527,  ..., -0.5486, -0.9128, -0.7535]])
n_rows: 2473

Class counts:
 0    309
1    226
2    384
3    393
4    396
5    381
6    384
Name: count, dtype: int64

Smallest value in X_train: -0.9781011343002319
Largest value in X_train: 0.9715812802314758


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [15]:
ck.look(first_rows, size=n_rows)


tensor([[-0.1391,  0.8480, -0.7045,  ..., -0.4990, -0.8969, -0.7742],
        [-0.5181,  0.0884, -0.4772,  ...,  0.7813,  0.5746, -0.3365],
        [-0.1337,  0.8543, -0.6965,  ..., -0.4501, -0.8997, -0.7674],
        [-0.1649,  0.8538, -0.6712,  ..., -0.3268, -0.8995, -0.7897],
        [-0.1198,  0.8906, -0.6527,  ..., -0.5486, -0.9128, -0.7535]])

Size of the data: 2473


✓ LOOK: A sample is shown and the size of the data is right.
Parts 2 of 8: LOAD ✓ | LOOK ✓ | SPLIT - | BATCH - | MODEL - | LOOP - | EVAL - | SUBMIT -

---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 7. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears.

The scorer of this task, in one line of scikit-learn. Run the cell; you will use `score(y_true, y_pred)` all afternoon.

In [16]:
def score(y_true, y_pred):
    """The score of a file: macro F1 over the classes."""
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


print(score([0, 1, 2, 2], [0, 2, 2, 2]))


0.6


## Task 3: What the scorer computes

The cell above is the scorer of this task. Which sentence says what it computes?

- **a** The share of texts whose class is right.
- **b** The F1 score of every class (from its precision and its recall), then the plain mean of those numbers: every class counts the same.
- **c** The F1 score of every class, then a mean in which a class counts as much as it has texts.

Store the letter in `formula`.

In [17]:
formula = 'b'

## Task 4: Two answers without a model

What would two answers that need no model score? `all_0` puts every text in class 0, and `all_6` puts every text in class 6. Score both on the labels `y_train` with `score` and store the two numbers in the dictionary `trivial` under these two names.

trivial: {'all_0': 0.03173462051966725, 'all_6': 0.0384019200960048}


## Task 5: What the metric rewards

A file is right on every text of the classes 0 to 4, but never says 5 or 6: it puts the texts of classes 5 and 6 into the classes 0 to 4. What can it score?

- **a** 1.0: every text of a class it knows is right.
- **b** Less than 5/7, about 0.71: classes 5 and 6 each score an F1 of 0 and count as much as any other class, and the misplaced texts also lower the precision of the classes 0 to 4.
- **c** About 0.95: only the texts of the two new classes are wrong.

Store the letter in `rewards`.

## Task 6: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, and `id_col`, the name of its id column. Read both from what you loaded; do not type the number.

In [20]:
# Your code here


{'n_rows': 700, 'id_col': 'id'}


## Task 7: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [22]:
facts = "The"

Run the cell to send your task reading. It answers with one line and shows no score.

In [23]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


Task reading received (minute 23).

---
# Part 3 -- Stage 1: the deployed classifier, frozen

Today's rule: the deployed classifier stays exactly as it is, and nothing new may be learned. You may compute averages of vectors and distances between vectors. Every checkpoint of this part names the stage `frozen`. Aim to hand in this stage's file within the first hour: it is your first file.

## Task 8: Hold out a validation set

Split the rows of `train` into a training part (80%) and a validation part (20%) with `train_test_split`, `random_state=0` and `stratify=y_train`, so that every class keeps its share. Keep the vectors and labels of the two parts in `X_tr`, `y_tr`, `X_va` and `y_va`, and the ids of their texts (from `train["id"]`) in `train_ids` and `val_ids`. Print the shapes of `X_tr` and `X_va`.

Syntax hint (the shape of the call, not the answer):

```python
part_a, part_b = train_test_split(np.arange(n), test_size=0.3, random_state=0, stratify=labels)
```

In [35]:
a, b = train_test_split(np.arange(len(X_train)), test_size = 0.2, random_state = 0, stratify = y_train.numpy())
X_tr, X_va = X_train[a], X_train[b]
y_tr, y_va = y_train[a], y_train[b]
train_ids = train["id"][a]
val_ids = train['id'][b]
print(X_tr.shape, X_va.shape)

torch.Size([1978, 768]) torch.Size([495, 768])


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [36]:
ck.split(train_ids=train_ids, val_ids=val_ids)


✓ SPLIT: The validation set shares no rows with the training set.
Parts 3 of 8: LOAD ✓ | LOOK ✓ | SPLIT ✓ | BATCH - | MODEL - | LOOP - | EVAL - | SUBMIT -

The vectors are already what a model takes. Run the cell: it checks `X_tr` and `y_tr` as your batch.

In [37]:
ck.batch(X_tr, y_tr)


✓ BATCH: Batch shape, size and data type are right.
Parts 4 of 8: LOAD ✓ | LOOK ✓ | SPLIT ✓ | BATCH ✓ | MODEL - | LOOP - | EVAL - | SUBMIT -

## Task 9: The deployed classifier

Load the deployed classifier into `head`: an `nn.Linear(768, 5)` that takes the weights of `classifier.pt` with `load_state_dict`. Then call `head.eval()`. Print a table with the true class of every text of `X_va` as rows and the class `head` gives it as columns (`pd.crosstab`).

Syntax hint (the shape of the call, not the answer):

```python
pd.crosstab(array_a, array_b, rownames=["a"], colnames=["b"])
```

Linear(in_features=768, out_features=5, bias=True)

Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [40]:
ck.model(head, X_va, stage="frozen")


✓ MODEL: The model runs on one batch and its output has the right shape and data type.
Parts 5 of 8: LOAD ✓ | LOOK ✓ | SPLIT ✓ | BATCH ✓ | MODEL ✓ | LOOP - | EVAL - | SUBMIT -

## Task 10: Seven classes under today's rule

Write `predict_frozen(X)`, which gives every row of a tensor of vectors `X` a class from 0 to 6. Today's rule holds: `head` stays as it is and nothing new is trained. You may use the labelled vectors of `X_tr`, averages of vectors and distances between them. Print what it gives the first ten rows of `X_va`, next to their true classes.

In [43]:
def predict_frozen(X):



## Task 11: Score the validation set

Store in `val_score` the `score` of `predict_frozen` on `X_va` against `y_va`: one number between 0 and 1. Print it.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="macro_f1", stage="frozen")


## Task 12: Write the submission file

Write `submission.csv` with the columns of `sample`: the `id` of every text of `test` and the class `predict_frozen` gives it. Leave the index out. Print its shape and its first rows.

Syntax hint (the shape of the call, not the answer):

```python
table.to_csv("name.csv", index=False)
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv", stage="frozen")


---
# Part 4 -- Debugging drill

A classmate began the next stage: one training step of a 7-class classifier. The cell stops with an error. Run the first cell: it starts the drill's clock. Then repair the classmate's cell until it runs and prints the loss, and run the check. The clock stops at the first check that passes. Read the error message before you change anything.

In [ ]:
ck.drill_start("bug-typ-1")


## Task 13: Repair the classmate's training step

Change as little as you can: the rows, the classifier and the loss stay as they are.

In [ ]:
torch.manual_seed(0)
drill_head = nn.Linear(768, 7)
drill_x = X_train[:32]
drill_y = y_train[:32].float()
drill_loss_fn = nn.CrossEntropyLoss()
drill_loss = drill_loss_fn(drill_head(drill_x), drill_y)
print(drill_loss)


In [ ]:
ck.drill_check("bug-typ-1", drill_loss)


---
# Part 5 -- Stage 2: the deployed classifier, fine-tuned

Next week a new version of the classifier may be deployed. So train the deployed classifier further on the task's data, starting from its own weights, and compare with Stage 1. Every checkpoint of this part names the stage `finetuned`.

## Task 14: A classifier for seven classes

Make `model`, an `nn.Linear(768, 7)` whose rows for the classes 0 to 4 start as the weights and the biases of `head` (copy them under `torch.no_grad()`); the rows of classes 5 and 6 keep their random start. Print the shape of `model(X_va)`.

Syntax hint (the shape of the call, not the answer):

```python
with torch.no_grad():
    big.weight[:2] = small.weight
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, X_va, stage="finetuned")


## Task 15: The training loop

Train `model` on `X_tr` and `y_tr`: a `DataLoader` of mini-batches over a `TensorDataset`, `nn.CrossEntropyLoss`, the Adam optimizer, and the loop written out, for a number of epochs of your choice. Append the loss of every step to the list `losses`.

Syntax hint (the shape of the call, not the answer):

```python
loader = DataLoader(TensorDataset(inputs, targets), batch_size=32, shuffle=True)
```

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(losses, stage="finetuned")


## Task 16: Score the validation set again

Store in `val_score_ft` the `score` of `model` on `X_va` (for every row, the class with the highest output) against `y_va`. Print it next to `val_score` of Stage 1.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score_ft, metric="macro_f1", stage="finetuned")


## Task 17: Write the fine-tuned file

Write `submission.csv` again, now with the classes `model` gives the texts of `test`.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv", stage="finetuned")


---
# Part 6 -- Challenge: your best file

You have a file of each stage on the board. Now make the fine-tuned one better, one change at a time, and let the validation set decide what stays.

## Task 18: Hand in your best file

Compare at least two settings on `X_va` (for example the number of epochs or the learning rate), each trained on `X_tr` from the deployed weights. Then train the best one on every labelled text, `X_train` and `y_train`, keep it in `final_model`, and write `submission.csv` again. Your last accepted file of this stage is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv", stage="finetuned")


---
## Task 19: Reflect

Which classes did your Stage 1 answer mix up most (look at a `pd.crosstab` of `y_va` against `predict_frozen(X_va)`)? How much did fine-tuning add to the validation score, and what did today's rule cost? When would you still choose the frozen way in a real product?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task**: the scorer read as code, two answers without a model, what a missing class costs in macro F1, and the format of the file, all before any model.
- **Pretrained vectors**: 768 numbers per text from mBERT, loaded from a file and used without the text.
- **A frozen model**: the deployed classifier kept as it is, with labelled neighbours beside it for what it never learned.
- **Debugging**: a data-type error read, located and repaired against the clock.
- **Fine-tuning**: the same classifier grown to seven classes from its own weights and trained further in a loop written out, then compared with the frozen stage on the validation set.

---

Made By **Sattam Altwaim**